# 04 — Feature Engineering and Selection

The goal of this stage is to prepare the account-level dataset for clustering.

We will assess feature quality, select relevant features, apply appropriate transformations, and scale the data.

## A. Feature Quality Assessment

We first examine the account-level dataset for missing values, invalid values, and duplicate account profiles before performing feature selection.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the account-level dataset
DATA_PATH = Path("data_processed/df_account_features_eda.csv.gz")

df_features = pd.read_csv(
    DATA_PATH,
    dtype={
        "ioCountry": "string",
        "accountid": "string"
    }
)

account_keys = ["ioCountry", "accountid"]

# Numerical features
feature_cols = [
    col for col in df_features.columns
    if col not in account_keys
]

print("Account profiles:", len(df_features))
print("Total attributes:", len(df_features.columns))
print("Numerical features:", len(feature_cols))

# Check duplicate account profiles
print(
    "\nDuplicate account profiles:",
    df_features.duplicated(subset=account_keys).sum()
)

# Check missing and infinite values
quality_summary = pd.DataFrame({
    "Missing": df_features[feature_cols].isna().sum(),
    "Missing (%)": (
        df_features[feature_cols].isna().mean() * 100
    ).round(2),
    "Infinite": np.isinf(
        df_features[feature_cols]
    ).sum()
})

display(quality_summary)

Account profiles: 86525
Total attributes: 16
Numerical features: 14

Duplicate account profiles: 0


,Missing,Missing (%),Infinite
post_count,0,0.0,0
posting_intensity,0,0.0,0
follower_count,4,0.0,0
following_count,3,0.0,0
account_age_days,0,0.0,0
activity_lifespan_days,0,0.0,0
repost_ratio,0,0.0,0
reply_ratio,0,0.0,0
mention_ratio,0,0.0,0
hashtag_ratio,0,0.0,0


In [2]:
# Features containing missing values
missing_features = ["follower_count", "following_count"]

# Replace missing values with each feature's median
for feature in missing_features:
    median_value = df_features[feature].median()

    df_features[feature] = df_features[feature].fillna(
        median_value
    )

    print(f"{feature}: filled missing values with {median_value:.2f}")

# Verify
print("\nRemaining missing values:", df_features[feature_cols].isna().sum().sum())

follower_count: filled missing values with 385.00
following_count: filled missing values with 406.00

Remaining missing values: 0


#### Feature Quality — Results

The dataset contains 86,525 account profiles and 14 numerical features, with no duplicate profiles or infinite values.

Only 4 follower counts and 3 following counts are missing. These values are replaced using the median of their respective features, which is appropriate given their skewed distributions.

All numerical features are now ready for further processing.

## B. Feature Selection

We evaluate the 14 numerical features to identify potential redundancy, unreliable features, and attributes that may not contribute useful information to clustering.

### B.1 — Correlated Features

We examine strongly correlated feature pairs to determine whether they provide overlapping information.
Highly correlated features will be reviewed before deciding which attributes to retain for clustering.
In distance-based clustering, such as K-means, including highly related features can give excessive importance to the same underlying behavior.

In [3]:
# Calculate Spearman correlations between numerical features
corr_matrix = df_features[feature_cols].corr(method="spearman")

# Keep only unique feature pairs (upper triangle)
upper = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape, dtype=bool),
        k=1
    )
)

# Convert correlations into a table
correlation_pairs = (
    upper.stack()
    .reset_index()
)

correlation_pairs.columns = [
    "Feature 1",
    "Feature 2",
    "Correlation"
]

# Calculate absolute correlations
correlation_pairs["Absolute Correlation"] = (
    correlation_pairs["Correlation"].abs()
)

# Display strongly correlated pairs
strong_pairs = correlation_pairs[
    correlation_pairs["Absolute Correlation"] >= 0.65
].sort_values("Absolute Correlation", ascending=False)

display(strong_pairs.round(3))

,Feature 1,Feature 2,Correlation,Absolute Correlation
92,repost_ratio,mention_ratio,0.817,0.817
31,follower_count,following_count,0.700,0.700


#### B.1 — Correlation Results

Two strongly correlated feature pairs were identified:

- `repost_ratio` and `mention_ratio` (0.817).
- `follower_count` and `following_count` (0.700).

These pairs may contain overlapping information. However, they represent different account characteristics, so we will retain them initially and reconsider their usefulness when evaluating clustering results.

In [4]:
# Identify accounts with exactly 99 posts
exactly_99 = df_features["post_count"] == 99

# Check whether their observed activity occurred on one day
same_day = df_features["activity_lifespan_days"] == 0

print("Total account profiles:", len(df_features))

print(
    "Accounts with exactly 99 posts:",
    exactly_99.sum()
)

print(
    "Percentage of accounts with 99 posts:",
    round(exactly_99.mean() * 100, 2), "%"
)

print(
    "Accounts with 99 posts and 0-day activity lifespan:",
    (exactly_99 & same_day).sum()
)

print(
    "Accounts with posting intensity exactly 99:",
    (df_features["posting_intensity"] == 99).sum()
)

Total account profiles: 86525
Accounts with exactly 99 posts: 11544
Percentage of accounts with 99 posts: 13.34 %
Accounts with 99 posts and 0-day activity lifespan: 11517
Accounts with posting intensity exactly 99: 11558


#### B.2 — Posting Activity Feature Assessment

We identified 11,544 accounts (13.34%) with exactly 99 observed posts, of which 11,517 have a zero-day activity lifespan.

This unusual concentration may reflect a data-collection or sampling effect, potentially affecting `post_count` and `posting_intensity`.

We will evaluate clustering with and without these features to determine whether they influence the resulting clusters.

In [5]:
# All 14 numerical features
features_all = feature_cols.copy()

# Exclude posting intensity
features_no_intensity = [
    col for col in feature_cols
    if col != "posting_intensity"
]

# Exclude both posting intensity and post count
features_no_activity = [
    col for col in feature_cols
    if col not in ["post_count", "posting_intensity"]
]

print("All features:", len(features_all))
print("Without posting intensity:", len(features_no_intensity))
print("Without both activity measures:", len(features_no_activity))

All features: 14
Without posting intensity: 13
Without both activity measures: 12


Now we have three candidate feature sets for clustering

## C. Feature Transformation

We examine the distributions of numerical features to identify highly skewed attributes.
Highly skewed features may require transformations to reduce the influence of extreme values before scaling and clustering.

In [6]:
# Examine feature distributions and skewness
skew_summary = pd.DataFrame({
    "Min": df_features[feature_cols].min(),
    "Median": df_features[feature_cols].median(),
    "Max": df_features[feature_cols].max(),
    "Skewness": df_features[feature_cols].skew()
})

# Sort by absolute skewness
skew_summary = skew_summary.reindex(
    skew_summary["Skewness"].abs()
    .sort_values(ascending=False)
    .index
)

display(skew_summary.round(2))

,Min,Median,Max,Skewness
post_count,1.0,23.00,265448.00,166.61
following_count,0.0,406.00,2301841.00,84.06
follower_count,0.0,385.00,19912528.00,54.55
activity_lifespan_days,0.0,0.00,3288.00,6.88
unique_applications,1.0,1.00,69.00,6.11
avg_post_length,2.0,132.33,2054.02,4.41
unique_languages,0.0,1.00,40.00,4.30
reply_ratio,0.0,0.00,1.00,2.17
posting_intensity,0.0,11.58,283.00,1.13
account_age_days,0.0,750.00,4493.00,0.94


### C.2 — Log Transformation

Several numerical features are strongly right-skewed, meaning that a small number of accounts have extremely large values.

We apply the log1p transformation to selected features to reduce the influence of extreme values.

The transformed distributions will be evaluated before standardization.

In [7]:
# Features selected for log transformation
log_features = [
    "post_count",
    "follower_count",
    "following_count",
    "activity_lifespan_days",
    "avg_post_length",
    "unique_languages",
    "unique_applications"
]

# Preserve the original feature values
df_transformed = df_features.copy()

# Apply log1p transformation
df_transformed[log_features] = np.log1p(
    df_transformed[log_features]
)

# Compare skewness before and after
skew_comparison = pd.DataFrame({
    "Before": df_features[log_features].skew(),
    "After": df_transformed[log_features].skew()
})

display(skew_comparison.round(2))

,Before,After
post_count,166.61,0.32
follower_count,54.55,-0.13
following_count,84.06,-0.88
activity_lifespan_days,6.88,2.19
avg_post_length,4.41,-0.45
unique_languages,4.30,1.34
unique_applications,6.11,1.93


#### C.2 — Log Transformation Results

Log transformation substantially reduced the skewness of the selected features, particularly post counts, follower counts, and following counts.

Some features remain moderately skewed, but their extreme values have less influence.

We retain the transformations and proceed to standardization before clustering.

## D. Feature Standardization

We apply Z-score standardization to the transformed numerical features.

This centers each feature around zero with a standard deviation of one, making their scales comparable for distance-based clustering.

The account identifiers are preserved without scaling.

In [8]:
from sklearn.preprocessing import StandardScaler

In [9]:
# Preserve transformed features and account identifiers
df_scaled = df_transformed.copy()

# Standardize the 14 numerical features
scaler = StandardScaler()

df_scaled[feature_cols] = scaler.fit_transform(
    df_transformed[feature_cols]
)

# Verify standardization
scaling_summary = pd.DataFrame({
    "Mean": df_scaled[feature_cols].mean(),
    "Std": df_scaled[feature_cols].std(ddof=0)
})

display(scaling_summary.round(3))

print("Scaled dataset shape:", df_scaled.shape)

,Mean,Std
post_count,-0.0,1.0
posting_intensity,0.0,1.0
follower_count,-0.0,1.0
following_count,-0.0,1.0
account_age_days,0.0,1.0
activity_lifespan_days,-0.0,1.0
repost_ratio,0.0,1.0
reply_ratio,-0.0,1.0
mention_ratio,0.0,1.0
hashtag_ratio,-0.0,1.0


Scaled dataset shape: (86525, 16)


### D.2 — Correlation After Transformation

We examine Pearson correlations between the transformed and standardized numerical features.

This helps identify strong linear relationships in the feature space that will be used for distance-based clustering.

Highly correlated features may be reconsidered when comparing clustering configurations.

In [13]:
# Pearson correlation after transformation and scaling
pearson_corr = df_scaled[feature_cols].corr(method="pearson")

# Keep unique feature pairs
upper = pearson_corr.where(
    np.triu(
        np.ones(pearson_corr.shape, dtype=bool),
        k=1
    )
)

# Create correlation table
pearson_pairs = (
    upper.stack()
    .reset_index()
)

pearson_pairs.columns = [
    "Feature 1",
    "Feature 2",
    "Pearson Correlation"
]

pearson_pairs["Absolute Correlation"] = (
    pearson_pairs["Pearson Correlation"].abs()
)

# Show the strongest relationships
display(
    pearson_pairs.sort_values(
        "Absolute Correlation",
        ascending=False
    ).head(10).round(3)
)

,Feature 1,Feature 2,Pearson Correlation,Absolute Correlation
92,repost_ratio,mention_ratio,0.794,0.794
31,follower_count,following_count,0.640,0.640
12,post_count,unique_languages,0.605,0.605
1,post_count,posting_intensity,0.529,0.529
151,url_ratio,avg_post_length,0.492,0.492
5,post_count,activity_lifespan_days,0.488,0.488
123,mention_ratio,avg_post_length,0.445,0.445
83,activity_lifespan_days,unique_applications,0.393,0.393
13,post_count,unique_applications,0.387,0.387
95,repost_ratio,avg_post_length,0.387,0.387


In [14]:
# Define the three candidate feature sets
candidate_feature_sets = {
    "all_features": features_all,
    "no_intensity": features_no_intensity,
    "no_activity": features_no_activity,
    "no_mention": [
        col for col in features_all
        if col != "mention_ratio"
    ]
}

# Create clustering matrices from standardized features
X_candidates = {
    name: df_scaled[columns].astype("float32")
    for name, columns in candidate_feature_sets.items()
}

# Keep account identifiers separately
account_ids = df_scaled[account_keys].copy()

# Verify the candidate matrices
for name, X in X_candidates.items():
    print(
        f"{name}: {X.shape} | "
        f"Missing values: {X.isna().sum().sum()}"
    )

all_features: (86525, 14) | Missing values: 0
no_intensity: (86525, 13) | Missing values: 0
no_activity: (86525, 12) | Missing values: 0
no_mention: (86525, 13) | Missing values: 0


## E. Final Feature Dataset

The account-level features have been cleaned, transformed, and standardized.

We prepared four candidate feature sets containing 14, 13, 12, and 13 numerical features to evaluate the influence of potentially unreliable or redundant attributes.

We save the complete standardized dataset, preserving account identifiers separately from the clustering features. The different feature configurations will be compared during clustering.

In [12]:
# Save the complete standardized dataset
OUTPUT_PATH = Path("data_processed/df_clustering_features.csv.gz")

df_scaled.to_csv(
    OUTPUT_PATH,
    index=False,
    compression="gzip"
)

print("Dataset saved:", OUTPUT_PATH)
print("Account profiles:", len(df_scaled))
print("Numerical features:", len(feature_cols))
print("Total columns:", len(df_scaled.columns))

Dataset saved: data_processed\df_clustering_features.csv.gz
Account profiles: 86525
Numerical features: 14
Total columns: 16
